# Group 7: NFL Ad Package Optimization (MVP)

**What this is:** our minimum viable model, for the **next group meeting**. It is *not* the final project.
The goal is for all of us to walk through the model and code together, make sure we understand every part, and decide what comes next.

It builds on the step notebooks (`Group7_Step1` through `Group7_Step3d`), with three fixes so it meets the rubric (see **Changes from Step 3d** at the end).

### Notes to remember

1. **Use the `.xlsx` data file, not the `.csv`.** `NFL Ad PKG MAIN.xlsx` has all 5 tabs. The `.csv` only has one tab, and it isn't the raw data.
2. **Keep `NFL Ad PKG MAIN.xlsx` in the same folder as this notebook.** The code opens it by name.
3. **`openpyxl` must be installed.** pandas uses it to read `.xlsx` files. Anaconda includes it, as it does `matplotlib`.
4. **Run the cells in order, top to bottom.** Part 1 creates the `clean_data/` folder that Part 2 reads.
5. **Never edit the raw spreadsheet.** The rubric asks us to submit it exactly as Ed provided it. Any change goes in the code.
6. **The sheet name `'Schedule Release 1-Min Qualifer'` is misspelled on purpose.** That's how it's spelled in Ed's file, so the code must match.

### How this notebook is organized

| Part | What it does |
| --- | --- |
| The problem | The story in plain words |
| 1. Raw data → clean data | Reads Ed's spreadsheet, checks it, saves clean CSV files |
| 2. Load clean data | The model reads only the clean files |
| 3. Team assumptions | Numbers that aren't in the spreadsheet, in one place |
| 4. The model | The math formulation, then the code |
| 5. Solve | Traces the trade-off curve |
| 6. Results | Table, max-profit plan, and chart |
| Discussion | Balancing the objectives, why this is the minimum, what we could remove, decisions for the meeting |

## The problem

A CBS affiliate sells a season-long ad package: one 30-second ad in each of 27 NFL games, for **\$108,000**. It's July, and the station must decide **which prospects to sell to, what audience to guarantee each one, and what discount to give.**

- **Customers care about CPM** (cost per thousand households). Each prospect has a goal CPM and won't pay more than that.
- **There are two ways to lower a customer's CPM:**
  - **Guarantee more audience.** It's the same price for more promised viewers, but if the station falls short, it owes a **makegood**.
  - **Give a discount.** It's the same audience for less money: a certain loss of revenue, but no risk.
- **The audience is uncertain in two ways:**
  - **Dud games:** the August schedule includes games with low-interest teams, which lose 30-40% of their audience.
  - **Nielsen's new 1-minute rule** raises ratings by 15%, 20%, or 25% (probabilities 50% / 35% / 15%).
- **Makegoods cost more for some customers.** Each customer is owed 2x, 3x, or 4x the value of the missing audience.

**The two goals conflict:** more revenue means selling more, with bigger guarantees, which means more makegood risk.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
import course_utils  # course helpers (print_status, print_model, ...)

## Part 1: Raw data → clean data

The rubric asks us to submit the **raw data as we got it**, the **clean data the model uses**, and **the code that turns one into the other**. This cell is that code.

It reads four blocks from Ed's spreadsheet, checks each one against the spreadsheet's own totals (the `assert` lines stop the notebook if anything doesn't match), and saves four clean files to `clean_data/`:

| Clean file | From | What's in it |
| --- | --- | --- |
| `package.csv` | Pre-Schedule Release + Schedule Release 1-Min Qualifier | Price (\$108,000), July baseline audience (1,631.3), August revised audience (1,465.9) |
| `schedule.csv` | Schedule Release 1-Min Qualifier, rows 13-39 | The 27 real games, before and after the dud cuts (for traceability) |
| `scenarios.csv` | Schedule Release 1-Min Qualifier, T6:U8 | The 3 Nielsen increases and their probabilities |
| `prospects.csv` | Prospect Data Table, rows 4-33 | The 30 prospects: ID, name, goal CPM, penalty multiplier |

Audiences are in **thousands of households ("000s")**, matching the spreadsheet. CPM is dollars per thousand, so 000s × CPM = dollars.

In [ ]:
# ============================================================
# PART 1: RAW DATA -> CLEAN DATA
# ============================================================

# RAW_FILE is Ed's spreadsheet, exactly as Ed provided it.
# We only read from it; we never edit it.
# This cell pulls out what the model needs, checks it against
# the spreadsheet's own totals, and saves clean CSV files.

RAW_FILE = "NFL Ad PKG MAIN.xlsx"
CLEAN_DIR = "clean_data"

os.makedirs(CLEAN_DIR, exist_ok=True)


# ------------------------------------------------------------
# A. The package: 'Pre-Schedule Release', rows 13-17
# ------------------------------------------------------------

# One row per game type. What the station knows in July.
# col C = game type, col E = number of games,
# col F = rate per spot ($), col H = audience per game (000s)

df_games = pd.read_excel(
    RAW_FILE,
    sheet_name="Pre-Schedule Release",
    header=None,
    skiprows=12,
    nrows=5,
    usecols="C,E,F,H"
)

df_games.columns = ["game_type", "num_games", "rate", "audience_000s"]

package_price = (df_games["num_games"] * df_games["rate"]).sum()
baseline_000s = (df_games["num_games"] * df_games["audience_000s"]).sum()

assert df_games["num_games"].sum() == 27
assert package_price == 108_000
assert abs(baseline_000s - 1631.3) < 0.001


# ------------------------------------------------------------
# B. The real schedule: 'Schedule Release 1-Min Qualifer', rows 13-39
# ------------------------------------------------------------

# One row per actual game, released in August. Games with
# low-interest teams are already cut 30% (one team) or 40% (two).
# col C = game ID, col D = game name, col F = rate per spot ($),
# col H = original audience (000s), col I = revised audience (000s)

df_sched = pd.read_excel(
    RAW_FILE,
    sheet_name="Schedule Release 1-Min Qualifer",
    header=None,
    skiprows=12,
    nrows=27,
    usecols="C,D,F,H,I"
)

df_sched.columns = ["game_id", "game", "rate", "original_000s", "revised_000s"]

revised_000s = df_sched["revised_000s"].sum()

assert len(df_sched) == 27
assert df_sched["rate"].sum() == 108_000
assert abs(df_sched["original_000s"].sum() - 1651.38) < 0.001
assert abs(revised_000s - 1465.938) < 0.001


# ------------------------------------------------------------
# C. Ratings scenarios: 'Schedule Release 1-Min Qualifer', T6:U8
# ------------------------------------------------------------

# col T = label, such as "15% Increase"; col U = probability

df_scen = pd.read_excel(
    RAW_FILE,
    sheet_name="Schedule Release 1-Min Qualifer",
    header=None,
    skiprows=5,
    nrows=3,
    usecols="T:U"
)

df_scen.columns = ["scenario", "probability"]

# "15% Increase" -> 0.15
df_scen["increase"] = df_scen["scenario"].str.split("%").str[0].astype(float) / 100

assert abs(df_scen["probability"].sum() - 1.0) < 0.000001


# ------------------------------------------------------------
# D. Prospects: 'Prospect Data Table', rows 4-33
# ------------------------------------------------------------

# col B = ID, col C = name, col F = goal CPM,
# col G = makegood penalty multiplier (2, 3, or 4)
# (cols D-E, salesperson and close ratio, are not used)

df_p = pd.read_excel(
    RAW_FILE,
    sheet_name="Prospect Data Table",
    skiprows=2,
    nrows=30,
    usecols="B,C,F,G"
)

df_p.columns = ["id", "prospect", "goal_cpm", "penalty"]

assert len(df_p) == 30


# ------------------------------------------------------------
# E. Save the clean files
# ------------------------------------------------------------

pd.DataFrame({
    "package_price": [package_price],
    "baseline_000s": [baseline_000s],
    "revised_000s": [revised_000s]
}).to_csv(f"{CLEAN_DIR}/package.csv", index=False)

df_sched.to_csv(f"{CLEAN_DIR}/schedule.csv", index=False)
df_scen.to_csv(f"{CLEAN_DIR}/scenarios.csv", index=False)
df_p.to_csv(f"{CLEAN_DIR}/prospects.csv", index=False)

print(f"Saved clean files to {CLEAN_DIR}/:", sorted(os.listdir(CLEAN_DIR)))

## Part 2: Load the clean data

From here on, the model reads **only** the clean files. This keeps the data separate from the model, as taught in class: to run the model on new data, you'd replace the clean files and change nothing below.

In [ ]:
# ============================================================
# PART 2: LOAD THE CLEAN DATA
# ============================================================

# From here on, the model reads ONLY the clean files.

df_package = pd.read_csv(f"{CLEAN_DIR}/package.csv")
df_scen = pd.read_csv(f"{CLEAN_DIR}/scenarios.csv")
df_p = pd.read_csv(f"{CLEAN_DIR}/prospects.csv")

package_price = df_package["package_price"].iloc[0]
baseline_000s = df_package["baseline_000s"].iloc[0]
revised_000s = df_package["revised_000s"].iloc[0]

print(f"Package price:  ${package_price:,.0f}")
print(f"Baseline 000s:  {baseline_000s:,.1f}   (July: generic games)")
print(f"Revised 000s:   {revised_000s:,.1f}   (August: dud games cut)")
print()
print(df_scen)
print()
df_p

## Part 3: Team assumptions

These numbers are **not** in Ed's spreadsheet. They're our choices, kept in one place so they're easy to find, change, and cite:

| Assumption | Value | Why |
| --- | --- | --- |
| Guarantee levels | 0% / 15% / 20% / 25% | Match the three Nielsen increases, plus none (**confirm with Ed**) |
| Discounts | 0% / 5% / 10% | Team decision, same as Plan B (**confirm with Ed**) |
| Packages available | 20 | 20 ad slots per game (from Ed) |
| Budget points on the curve | 10 | Enough to see the shape of the trade-off |
| Tie-breaker | 0.0001 | Among plans with equal revenue, prefer less risk (explained in Part 5) |

**Modeling assumptions (for the report):**
1. The makegood is valued at the CPM the customer actually paid, after the discount (**confirm with Ed**).
2. A prospect whose goal CPM is met buys the package. We don't model the chance that a sale falls through (the close ratio).
3. The station plans for the dud games, using the August schedule as its best estimate. In reality it sells in July, before it knows the duds. This is a simplification.
4. Shortfall is measured on the whole package, not game by game (**confirm with Ed**).

In [ ]:
# ============================================================
# PART 3: TEAM ASSUMPTIONS
# ============================================================

# Numbers that are NOT in Ed's spreadsheet. Kept in one place
# so they're easy to find, change, and cite in the report.

# Guarantee level for each package: audience promised above the
# July baseline. Chosen to match the 3 Nielsen increases, plus none.
guarantee = {
    "conservative": 0.00,
    "medium": 0.15,
    "premium": 0.20,
    "aggressive": 0.25
}

# Discount options off the $108,000 price
discounts = [0.00, 0.05, 0.10]

# Packages available to sell (20 ad slots per game, per Ed)
total_packages = 20

# Number of evenly spaced makegood budgets on the trade-off curve
num_points = 10

# Tie-breaker: among plans with the same revenue, prefer less risk.
# Worth 1/100th of a cent per dollar of risk, so it never changes
# which plan has the most revenue.
tie_breaker = 0.0001

## Part 4: The model

### In math

**Sets**
- $C$ = customers (30 prospects), indexed by $c$
- $P$ = packages (4 guarantee levels), indexed by $p$
- $D$ = discounts (3 options), indexed by $d$
- $S$ = Nielsen scenarios (3), indexed by $s$

**Parameters**
- $\text{price}_{pd} = 108{,}000 \times (1 - d)$: price of package $p$ at discount $d$
- $G_p = 1{,}631.3 \times (1 + g_p)$: audience guaranteed by package $p$ (000s)
- $\text{cpm}_{pd} = \text{price}_{pd} \,/\, G_p$: CPM the customer pays
- $\text{goal}_c$: customer $c$'s goal CPM
- $m_c$: customer $c$'s makegood multiplier (2, 3, or 4)
- $\pi_s$: probability of scenario $s$
- $A_s = 1{,}465.9 \times (1 + \text{increase}_s)$: actual audience delivered in scenario $s$ (000s)
- $E_{cpd} = m_c \times \text{cpm}_{pd} \times \sum_{s} \pi_s \max(G_p - A_s,\ 0)$: expected makegood cost if customer $c$ buys package $p$ at discount $d$

**Decision variable**
- $x_{cpd} \in \{0, 1\}$: 1 if customer $c$ buys package $p$ at discount $d$

**Objectives**
- Maximize expected revenue: $\displaystyle \sum_{c,p,d} \text{price}_{pd}\, x_{cpd}$
- Minimize expected makegood cost: $\displaystyle \sum_{c,p,d} E_{cpd}\, x_{cpd}$

**Constraints**
1. One package per customer: $\displaystyle \sum_{p,d} x_{cpd} \le 1 \quad \forall c$ (30 constraints)
2. CPM requirement: $\displaystyle \sum_{p,d} \text{cpm}_{pd}\, x_{cpd} \le \text{goal}_c \sum_{p,d} x_{cpd} \quad \forall c$ (30 constraints)
3. Station inventory: $\displaystyle \sum_{c,p,d} x_{cpd} \le 20$ (1 constraint)
4. Makegood budget: $\displaystyle \sum_{c,p,d} E_{cpd}\, x_{cpd} \le B$ (1 constraint; $B$ changes to trace the trade-off)

### In plain words

| Piece | What it says about the world |
| --- | --- |
| Decision | For each customer: which guarantee and which discount, or nothing |
| Family 1 | Each advertiser buys at most one package |
| Family 2 | The price per thousand a customer pays can't be above their goal. If they buy nothing, both sides are 0 |
| Family 3 | The station only has 20 packages to sell |
| Family 4 | The station's appetite for makegood risk has a limit |
| Objective 1 | Bring in as much revenue as possible |
| Objective 2 | Owe as little in makegoods as possible |

**Why Family 2 is written this way:** an earlier version said `x[c,p,d] <= eligible[c,p,d]`, but the rubric says bounds on individual variables don't count. Writing it per customer ("the CPM customer c pays ≤ their goal") says the same thing, gives the same answers, and is a real constraint across several variables.

**Rubric check:** the cell below prints Gurobi's own counts: 360 variables, 62 constraints, 4 families, 2 objectives.

In [ ]:
# ============================================================
# PART 4: THE MODEL
# ============================================================

# ============================================================
# 1. INDEX SETS
# ============================================================

customers = df_p["id"].tolist()          # 30 prospects
packages = list(guarantee.keys())        # 4 guarantee levels
scenarios = df_scen["scenario"].tolist() # 3 Nielsen increases
# discounts: 3 options (team assumptions cell)


# ============================================================
# 2. CUSTOMER PARAMETERS
# ============================================================

goal_cpm = dict(zip(df_p["id"], df_p["goal_cpm"]))

# Makegood multiplier: if we fall short, customer c is owed
# penalty[c] times the value of the missing audience.

underdelivery_penalty = dict(zip(df_p["id"], df_p["penalty"]))


# ============================================================
# 3. AD PACKAGE PARAMETERS
# ============================================================

# Audience we promise the customer (000s), based on the July baseline

guaranteed_000s = {}

for p in packages:
    guaranteed_000s[p] = baseline_000s * (1 + guarantee[p])

# Price and CPM depend on the package and the discount.
# A bigger guarantee or a bigger discount means a lower CPM.

price = {}
cpm = {}

for p in packages:
    for d in discounts:
        price[p, d] = package_price * (1 - d)
        cpm[p, d] = price[p, d] / guaranteed_000s[p]


# ============================================================
# 4. ACTUAL AUDIENCE OUTCOMES
# ============================================================

# The station plans for the dud games: actual audience starts
# from the REVISED audience (after the dud-game cuts).
#
# Actual audience = revised x (1 + Nielsen increase)

probability = dict(zip(df_scen["scenario"], df_scen["probability"]))
increase = dict(zip(df_scen["scenario"], df_scen["increase"]))

actual_000s = {}

for s in scenarios:
    actual_000s[s] = revised_000s * (1 + increase[s])


# ============================================================
# 5. CALCULATE EXPECTED MAKEGOOD COST
# ============================================================

# Shortfall (000s) x CPM the customer paid ($ per 000) = value
# of the missing audience in dollars. The customer is owed that
# value times their penalty multiplier.

expected_penalty = {}

for c in customers:
    for p in packages:
        for d in discounts:

            expected_penalty[c, p, d] = sum(
                probability[s] *
                max(guaranteed_000s[p] - actual_000s[s], 0) *
                cpm[p, d] *
                underdelivery_penalty[c]
                for s in scenarios
            )


# ============================================================
# 6. CREATE GUROBI MODEL
# ============================================================

model = gp.Model("NFL_Ad_Package_MVP")

model.Params.OutputFlag = 0   # we solve many times, so hide the solver log


# ============================================================
# 7. DECISION VARIABLES
# ============================================================

# x[c,p,d] = 1 if customer c receives package p at discount d
#            0 otherwise

x = model.addVars(
    customers,
    packages,
    discounts,
    vtype=GRB.BINARY,
    name="x"
)


# ============================================================
# 8. CONSTRAINTS
# ============================================================

# ------------------------------------------------------------
# Family 1: One package per customer
# Each customer buys at most one package (one guarantee level
# at one discount).
# ------------------------------------------------------------

model.addConstrs(
    (
        gp.quicksum(x[c, p, d] for p in packages for d in discounts) <= 1
        for c in customers
    ),
    name="one_package_per_customer"
)


# ------------------------------------------------------------
# Family 2: Customer's CPM requirement
# The CPM a customer pays cannot exceed their goal CPM.
# (If the customer buys nothing, both sides are 0.)
# ------------------------------------------------------------

model.addConstrs(
    (
        gp.quicksum(cpm[p, d] * x[c, p, d] for p in packages for d in discounts)
        <= goal_cpm[c] * gp.quicksum(x[c, p, d] for p in packages for d in discounts)
        for c in customers
    ),
    name="cpm_requirement"
)


# ------------------------------------------------------------
# Family 3: Station inventory
# We can sell at most 20 packages.
# ------------------------------------------------------------

model.addConstr(
    gp.quicksum(
        x[c, p, d]
        for c in customers
        for p in packages
        for d in discounts
    ) <= total_packages,
    name="advertising_spots"
)


# ============================================================
# 9. TWO OBJECTIVES
# ============================================================

# Objective 1: maximize expected revenue

total_revenue = gp.quicksum(
    price[p, d] * x[c, p, d]
    for c in customers
    for p in packages
    for d in discounts
)

# Objective 2: minimize expected makegood cost

total_penalty = gp.quicksum(
    expected_penalty[c, p, d] * x[c, p, d]
    for c in customers
    for p in packages
    for d in discounts
)


# ------------------------------------------------------------
# Family 4: Makegood budget
# Expected makegood cost cannot exceed the budget.
# We start with no limit, and change the budget in section 10.
# ------------------------------------------------------------

budget = model.addConstr(
    total_penalty <= GRB.INFINITY,
    name="makegood_budget"
)


# ============================================================
# RUBRIC CHECK (counts straight from Gurobi)
# ============================================================

model.update()

print(f"Decision variables:  {model.NumVars}")
print(f"Constraints:         {model.NumConstrs}  (none are bounds on a single variable)")
print("Constraint families: 4")
print("  1. one_package_per_customer  (30)")
print("  2. cpm_requirement           (30)")
print("  3. advertising_spots         (1)")
print("  4. makegood_budget           (1)")
print("Objectives:          2  (maximize revenue, minimize makegood cost)")

## Part 5: Solve

We can't maximize revenue *and* minimize risk at the same time, so we trace the **trade-off curve**: the most revenue possible for each level of risk.

- **Step A (the riskiest end):** maximize revenue with no budget.
- **Step B (Ed's view):** maximize revenue minus makegood cost (expected profit), with no budget.
- **Step C (the curve):** for 10 evenly spaced budgets between \$0 and the riskiest plan, plus the max-profit point, set the budget and maximize revenue. `budget.RHS = level` changes the budget constraint's right-hand side: same model, new limit.

**Why the tie-breaker:** several plans can tie for the most revenue. Aggressive and premium at full price both bring in \$108,000, but carry different risk. Without a tie-breaker, Gurobi picks one of the tied plans arbitrarily, and the riskiest end of the curve moves around (we saw \$707K one run and \$791K another). Subtracting `0.0001 × makegood cost` means "among plans with equal revenue, take the less risky one." It's too small to ever trade real revenue for risk.

In [ ]:
# ============================================================
# 10. SOLVE: TRACE THE TRADE-OFF CURVE
# ============================================================

# Step A: the riskiest end. Maximize revenue with no budget.
#         (The tie-breaker picks the least risky of the plans
#          that tie for the most revenue.)

model.setObjective(total_revenue - tie_breaker * total_penalty, GRB.MAXIMIZE)
model.optimize()

max_penalty = total_penalty.getValue()


# Step B: the max-profit point (Ed's view). Maximize expected
#         revenue minus expected makegood cost, with no budget.

model.setObjective(total_revenue - total_penalty, GRB.MAXIMIZE)
model.optimize()

best_revenue = total_revenue.getValue()
best_penalty = total_penalty.getValue()

best_plan = [(c, p, d) for (c, p, d) in x if x[c, p, d].X > 0.5]


# Step C: evenly spaced budgets from 0 to max_penalty, plus the
#         max-profit point. For each budget, maximize revenue.

budget_levels = [max_penalty * i / num_points for i in range(num_points + 1)]
budget_levels.append(best_penalty)
budget_levels.sort()

model.setObjective(total_revenue - tie_breaker * total_penalty, GRB.MAXIMIZE)

results = []

for level in budget_levels:

    budget.RHS = level

    model.optimize()

    revenue = total_revenue.getValue()
    penalty = total_penalty.getValue()

    row = {
        "budget": level,
        "revenue": revenue,
        "penalty": penalty,
        "profit": revenue - penalty,
        "sold": sum(1 for k in x if x[k].X > 0.5)
    }

    for p in packages:
        row[p] = sum(1 for (c, pp, d) in x if pp == p and x[c, pp, d].X > 0.5)

    results.append(row)

df_results = pd.DataFrame(results)

budget.RHS = GRB.INFINITY   # reset

## Part 6: Results

In [ ]:
# ============================================================
# 11. DISPLAY RESULTS
# ============================================================

print("TRADE-OFF: best revenue for each makegood budget")
print("=" * 75)
print(
    df_results.to_string(
        index=False,
        formatters={
            "budget": "${:,.0f}".format,
            "revenue": "${:,.0f}".format,
            "penalty": "${:,.0f}".format,
            "profit": "${:,.0f}".format
        }
    )
)


print("\nMAX-PROFIT PLAN (Ed's view)")
print("=" * 75)

for (c, p, d) in best_plan:

    print(
        f"{c}: "
        f"{p.capitalize():12} | "
        f"Discount: {d:>4.0%} | "
        f"Goal CPM: ${goal_cpm[c]:>5.2f} | "
        f"CPM: ${cpm[p, d]:>5.2f} | "
        f"Penalty: {underdelivery_penalty[c]}x | "
        f"Revenue: ${price[p, d]:>9,.0f} | "
        f"Expected Makegood: ${expected_penalty[c, p, d]:>9,.2f}"
    )

print("=" * 75)
print(f"Packages Sold:           {len(best_plan)} of {total_packages}")
print(f"Expected Revenue:        ${best_revenue:,.2f}")
print(f"Expected Makegood Cost:  ${best_penalty:,.2f}")
print(f"Expected Profit:         ${best_revenue - best_penalty:,.2f}")


# ------------------------------------------------------------
# Chart: revenue (vertical) against makegood cost (horizontal)
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    df_results["penalty"] / 1000,
    df_results["revenue"] / 1000,
    marker="o", markersize=8, linewidth=2,
    color="#2a78d6"
)

ax.plot(
    best_penalty / 1000,
    best_revenue / 1000,
    marker="o", markersize=12, linestyle="none",
    color="#eb6834", markeredgecolor="white", markeredgewidth=2
)

ax.annotate(
    f"Max profit (Ed's view)\n${(best_revenue - best_penalty) / 1000:,.0f}K profit",
    xy=(best_penalty / 1000, best_revenue / 1000),
    xytext=(25, -70), textcoords="offset points",
    color="#0b0b0b", fontsize=10,
    arrowprops=dict(arrowstyle="-", color="#52514e")
)

ax.set_title("Revenue vs. makegood risk", loc="left", fontsize=13, color="#0b0b0b")
ax.set_xlabel("Expected makegood cost ($ thousands)", color="#52514e")
ax.set_ylabel("Expected revenue ($ thousands)", color="#52514e")
ax.grid(color="#e6e5e0", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

### What the results show

**How to read the curve:** its steepness is the revenue gained for each extra dollar of makegood risk.

- **Left end (steep): risk pays well.** The first \$141K of risk buys \$686K of revenue, about \$5 per \$1 of risk. The model sells mostly **medium**, the safest guarantee customers will buy.
- **Middle: returns shrink.** More customers need bigger guarantees, so each dollar of risk buys less.
- **The max-profit point (\$504K of risk):** up to here, each extra dollar of risk still brings in more than a dollar of revenue.
- **Right end (flat): risk stops paying.** Going to the riskiest plan adds \$203K of risk for only \$70K of revenue. All 17 customers are already sold, so the extra revenue comes only from moving customers to **aggressive** at smaller discounts.

**Discounts are what make the curve.** Moving left from the riskiest end, customers shift from aggressive (11 → 4) to medium (2 → 10): from "big guarantee, full price" to "safe guarantee, a bit off."

**Why the curve isn't perfectly smooth:** each sale is yes or no, so the best plan can jump. Going from a \$494,831 budget to \$504,307 adds under \$15,000 of risk but \$75,600 of revenue: one more customer fits, and several switch options.

**Other things to notice:**
- **Nobody ever buys conservative.** At \$66.20 CPM (or \$59.58 with 10% off), it's above every prospect's goal. That's *why* the station needs guarantees at all.
- **17 of 20 packages sell at most.** The other 13 prospects have goals below the cheapest option (\$47.67), so the 20-package limit never binds with this data.

## How we balance the two objectives (proposed, for the group to decide)

The rubric says the balance is **our decision, and we must defend it.**

**Our proposal: the max-profit point** (\$1,690,200 revenue, \$504,307 expected makegood cost, \$1,185,893 expected profit).

**The defense, in one sentence:** up to this point, every extra dollar of makegood risk we accept brings in more than a dollar of revenue. Beyond it, extra risk costs more than it earns.

**Alternatives we could defend instead:**
- **A more cautious point**, such as the \$353K budget (\$1,377,000 revenue, 14 customers). It gives up about \$161K of expected profit for 30% less exposure. That's defensible if the station can't afford a bad year, for example if makegoods would crowd out inventory it needs to sell later.
- **A hard risk limit from management**, such as "expected makegoods under 20% of revenue." Then we'd read the answer off the curve.

## Why this is the minimum complexity

Every piece of the model is there because, without it, **the trade-off disappears or the rubric fails.** We tested each one:

| Piece | Without it | Keep? |
| --- | --- | --- |
| **Guarantees** (packages above 0%) | No makegood risk at all, so there's no second objective | **Required** |
| **CPM requirement** (Family 2) | The station sells 20 packages at full price with no guarantee: \$2,160,000 revenue, **\$0 risk**. The objectives stop conflicting. Also only 32 constraints, below the rubric's 50 | **Required** |
| **Discounts** | Rubric counts still pass (120 variables), but only 8 prospects can buy, and **the max-profit plan is simply "sell to everyone who qualifies"** (\$864,000 revenue, \$313,292 risk). There is nothing to balance: the only way to cut risk is to turn customers away | **Required for a real trade-off** |
| **Makegood budget** (Family 4) | We could only compute one point (max profit), not the curve the rubric's "two objectives" needs | **Required** |
| **One package per customer** (Family 1) | A customer could buy several packages, which isn't real | **Required** |
| **Dud games** (Step 3c) | Still works, but total risk is only \$215K, and we lose Ed's real-world problem | Strongly recommended |
| **Station inventory** (Family 3) | Same answers today (it doesn't bind), but only 3 families, the bare minimum | Recommended (see next section) |

**In short:** guarantees create the risk, the CPM rule forces the station to use them, discounts give it a way out, and the budget lets us map the trade-off. Remove any one of those, and there is no real decision left.

## If we wanted to simplify further

We tested each candidate. Here's what could go and still meet the rubric:

1. **Remove the conservative package (recommended if we simplify).** Nobody ever buys it, so removing it changes **nothing**: identical results, 270 variables instead of 360. The only loss is explanatory, because it shows *why* guarantees exist (no one pays \$66 CPM). We could mention it in the text instead of the code.
2. **Remove the dud games (Step 3c) (possible, with a trade-off).** The model is the same size, and it **removes the timeline simplification** (assumption 3), which the professor might question. But risk drops to \$215K, the story loses Ed's real-world problem, and the curve starts at \$1,009,800 (10 customers can buy medium with no risk at all). This is a real decision for the group.
3. **Remove the station inventory limit (not recommended).** Identical results today, but it leaves exactly 3 families and 61 constraints, right at the rubric's minimum with no margin. It's one line of code, and it's real.

**What we can't remove:** guarantees, the CPM requirement, discounts, the makegood budget, or one package per customer. Each one is either required by the rubric or required for the objectives to genuinely conflict.

## Decisions for the group meeting

1. **Balance point:** do we go with the max-profit point, or choose and defend a more cautious one?
2. **Dud games:** keep them (real risk, but a timeline simplification) or remove them (clean timeline, small risk)?
3. **Simplify?** Remove the conservative package from the code?
4. **Next extension, if any:** salesperson hours after the schedule comes out (Plan B's Stage 2)? It would address the timeline issue, but it's the most complex piece.
5. **Report plan:** who writes which section, and the data provenance paragraph (below).

### Questions for Ed (data provenance; these matter the most)

1. **Prospect table:** are the goal CPMs and 2x/3x/4x multipliers real station data (anonymized or rescaled)? The goal CPMs are identical to our original theoretical base model. If they were made up, whether by us or with AI help, the rubric doesn't allow them, and we need real ones.
2. **Where did each input come from?** The Nielsen probabilities (50/35/15%), the low-interest team list and the 30%/40% cuts, the rates, and the audience estimates. Are they management estimates, rate cards, or historical data?
3. **What was anonymized or rescaled** (market name, prospect names, numbers)? The report must say what changed and why.
4. **Permission:** can we share this data? If unsure, we must ask the professor **before Week 5**.
5. **Confirm the assumptions** in Part 3: guarantee levels, discounts, makegood valued at the discounted CPM, and package-total vs. game-by-game shortfall.

**Data provenance paragraph (template, to fill in once Ed answers):**

> The data comes from [Ed's full name]'s work as [role] at [type of agency], which gives [Ed] first-hand access to the station's NFL package sales. The spreadsheet `NFL Ad PKG MAIN.xlsx` was [exported from / assembled from] [source system or documents] on [date]. The rates and audience estimates come from [rate card / Nielsen data / station projections]; the prospect goal CPMs and makegood multipliers come from [source]; the Nielsen increase probabilities are [management's estimates / historical]. To protect confidential information, we [anonymized the market and advertiser names / rescaled ...]. Obtaining the data took [effort: time, approvals, who was involved].

## Changes from Step 3d

1. **Family 2 rewritten per customer** (`x <= eligible` was a bound, which the rubric doesn't count). Same answers; the model goes from 32 to **62** non-trivial constraints.
2. **Tie-breaker** added so the riskiest end of the curve is well defined (the least risky of the plans tied for the most revenue).
3. **Data separated from the model:** Part 1 saves clean CSV files; the model reads only those. Team assumptions moved to their own cell (Part 3).
4. **Eligibility dictionary removed.** Family 2 does its job directly.
5. **Added:** the math formulation, a printed rubric check, and the max-profit plan by customer.

With the tie-breaker, a few middle points on the curve are slightly less risky than in Step 3d for the same revenue (for example, \$352,479 vs. \$352,889 at the \$353K budget). The max-profit plan is identical.